In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.10.0+cpu
transformers: 5.2.0
cuda: False


In [2]:
OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"

df_wazuh = pd.read_parquet(parquet_path)
print("Loaded:", df_wazuh.shape)

def map_priority(level):
    if pd.isna(level):
        return None
    level = float(level)
    if level <= 3:
        return "low"
    elif level <= 6:
        return "medium"
    elif level <= 10:
        return "high"
    else:
        return "critical"

df = df_wazuh.copy()
df["y_priority"] = df["rule_level"].apply(map_priority)

df_ml = df.dropna(subset=["full_log", "y_priority"]).copy()
df_ml["full_log"] = df_ml["full_log"].astype("string").fillna("")
print("After dropna:", df_ml.shape)
display(df_ml["y_priority"].value_counts())


Loaded: (2600263, 14)
After dropna: (2293628, 15)


y_priority
medium      1873973
low          286087
high         131901
critical       1667
Name: count, dtype: int64

In [3]:
RANDOM_STATE = 42
N_SAMPLE = 200_000  # как в LSTM

df_s = df_ml.sample(n=min(N_SAMPLE, len(df_ml)), random_state=RANDOM_STATE).copy()
print("Sampled:", df_s.shape)
display(df_s["y_priority"].value_counts())

X = df_s["full_log"].tolist()
y_text = df_s["y_priority"].tolist()

le = LabelEncoder()
y = le.fit_transform(y_text)
print("Classes:", list(le.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", len(X_train), "Test:", len(X_test))
print("Test class counts:", np.bincount(y_test))


Sampled: (200000, 15)


y_priority
medium      163140
low          25010
high         11689
critical       161
Name: count, dtype: int64

Classes: [np.str_('critical'), np.str_('high'), np.str_('low'), np.str_('medium')]
Train: 160000 Test: 40000
Test class counts: [   32  2338  5002 32628]


In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_dict({"text": X_train, "label": y_train})
test_ds  = Dataset.from_dict({"text": X_test,  "label": y_test})

def tok_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
test_ds  = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

train_ds.set_format("torch")
test_ds.set_format("torch")

train_ds[0].keys()


Map: 100%|██████████| 40000/40000 [00:02<00:00, 17932.59 examples/s]


dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])

In [5]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)

# cap (как мы делали для LSTM), чтобы не уходило в безумие из-за tiny critical
cw = np.clip(cw, 0.5, 10.0)

class_weights = np.array([cw[i] for i in range(len(cw))], dtype=np.float32)
print("class_weights:", {le.classes_[i]: float(class_weights[i]) for i in range(len(class_weights))})


class_weights: {np.str_('critical'): 10.0, np.str_('high'): 4.277617454528809, np.str_('low'): 1.9992003440856934, np.str_('medium'): 0.5}


In [6]:
import inspect
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "macro_f1": macro_f1}

# Совместимый TrainingArguments (как в прошлом файле)
sig = inspect.signature(TrainingArguments.__init__).parameters
kwargs = dict(
    output_dir="hf_out_imbal_200k",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,          # ⚡ на 200k CPU — начинаем с 1 эпохи
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
)

if "evaluation_strategy" in sig:
    kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in sig:
    kwargs["eval_strategy"] = "epoch"

if "save_strategy" in sig:
    kwargs["save_strategy"] = "epoch"

if "load_best_model_at_end" in sig:
    kwargs["load_best_model_at_end"] = True
if "metric_for_best_model" in sig:
    kwargs["metric_for_best_model"] = "macro_f1"
if "greater_is_better" in sig:
    kwargs["greater_is_better"] = True
if "report_to" in sig:
    kwargs["report_to"] = "none"
if "fp16" in sig:
    kwargs["fp16"] = False

training_args = TrainingArguments(**kwargs)

w = torch.tensor(class_weights, dtype=torch.float32)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    class_weights=w,
)

print("Ready. Args:", kwargs)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1621.19it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Ready. Args: {'output_dir': 'hf_out_imbal_200k', 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 16, 'num_train_epochs': 1, 'learning_rate': 2e-05, 'weight_decay': 0.01, 'logging_steps': 100, 'eval_strategy': 'epoch', 'save_strategy': 'epoch', 'load_best_model_at_end': True, 'metric_for_best_model': 'macro_f1', 'greater_is_better': True, 'report_to': 'none', 'fp16': False}


In [7]:
trainer.train()


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.699101,0.883335,0.940725,0.491210


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.79it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=20000, training_loss=0.7370945795059204, metrics={'train_runtime': 24214.8085, 'train_samples_per_second': 6.608, 'train_steps_per_second': 0.826, 'total_flos': 5298884935680000.0, 'train_loss': 0.7370945795059204, 'epoch': 1.0})